# End-to-End ETL Pipeline (Local)

Runs the full WeatherSpeak PH pipeline locally using `modal_etl/core/` modules.

**Pipeline:** PDF → OCR markdown + metadata → radio scripts + TTS text → MP3

The same `run_step1 / run_step2 / run_step3` functions are used by the Modal ETL in production.
Output mirrors the Modal Volume layout: `output/{stem}/ocr.md`, `radio_{lang}.md`, etc.

Set `force=True` in any step cell to re-run that step even if outputs already exist.

In [1]:
import sys
from pathlib import Path

# Make modal_etl importable from notebook directory
sys.path.insert(0, str(Path.cwd().parent))

from modal_etl.core.ocr import run_step1
from modal_etl.core.scripts import run_step2
from modal_etl.core.tts import run_step3

In [2]:
# ── Configuration ────────────────────────────────────────────────────────
STEM          = "PAGASA_25-TC22_Verbena_TCB#24"
PDF_PATH      = Path("../data/bulletin-archive/archive/pagasa-25-TC22") / f"{STEM}.pdf"
OUTPUT_DIR    = Path("10-etl-e2e/output")
OLLAMA_URL    = "http://localhost:11434"
LANGUAGES     = ["ceb", "tl", "en"]
TTS_MODELS_DIR = Path.home() / ".cache" / "huggingface" / "hub"
FORCE_RUN       = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"PDF:        {PDF_PATH}  (exists={PDF_PATH.exists()})")
print(f"Output dir: {OUTPUT_DIR.resolve()}")
print(f"Ollama URL: {OLLAMA_URL}")

PDF:        ../data/bulletin-archive/archive/pagasa-25-TC22/PAGASA_25-TC22_Verbena_TCB#24.pdf  (exists=True)
Output dir: /Users/josereyes/Dev/gemma4-hackathon/notebooks/10-etl-e2e/output
Ollama URL: http://localhost:11434


## Multi-Bulletin Validation (4 PDFs)

Runs Step 1 (OCR + metadata) across 4 representative bulletins to validate the
updated two-pass narrative extraction. Set `STEMS` to the 4 PDFs open in your IDE.

In [ ]:
from modal_etl.core.ocr import run_step1
from pathlib import Path
import json

OLLAMA_URL = "http://localhost:11434"
ARCHIVE_ROOT = Path("../data/bulletin-archive/archive")
OUTPUT_DIR = Path("10-etl-e2e/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Set these to the 4 PDFs open in your IDE
STEMS = [
    ("pagasa-20-19W", "PAGASA_20-19W_Pepito_SWB#01"),
    ("pagasa-22-TC02", "PAGASA_22-TC02_Basyang_TCA#01"),
    ("pagasa-25-TC22", "PAGASA_25-TC22_Verbena_TCB#24"),
    ("pagasa-26-TC02", "PAGASA_26-TC02_Basyang_TCB#01"),
]

for archive_subdir, stem in STEMS:
    pdf_path = ARCHIVE_ROOT / archive_subdir / f"{stem}.pdf"
    print(f"\n{'='*60}")
    print(f"Processing: {stem}")
    print(f"PDF exists: {pdf_path.exists()}")
    stem_dir = run_step1(pdf_path, OUTPUT_DIR, ollama_url=OLLAMA_URL, force=True)
    print(f"Done → {stem_dir}")

In [ ]:
# Validate outputs for each stem
for _, stem in STEMS:
    stem_dir = OUTPUT_DIR / stem
    print(f"\n{'='*60}")
    print(f"Stem: {stem}")

    # Check ocr.md does NOT contain forecast table rows
    ocr_md = (stem_dir / "ocr.md").read_text(encoding="utf-8")
    table_leaked = "12-Hour Forecast" in ocr_md or "24-Hour Forecast" in ocr_md
    print(f"  ocr.md: {len(ocr_md)} chars | forecast table leaked: {table_leaked}")

    # Check forecast_table.md exists and has content
    ft_md = (stem_dir / "forecast_table.md").read_text(encoding="utf-8")
    print(f"  forecast_table.md: {len(ft_md)} chars")

    # Check new fields in metadata.json
    meta = json.loads((stem_dir / "metadata.json").read_text(encoding="utf-8"))
    print(f"  wind_extent:  {meta.get('wind_extent')}")
    print(f"  land_hazards: {meta.get('land_hazards')}")
    print(f"  track_outlook: {meta.get('track_outlook')}")
    fp_count = len(meta.get("forecast_positions", []))
    print(f"  forecast_positions: {fp_count} rows")

## Step 1: OCR — PDF → Markdown + Metadata

Sends each PDF page to Gemma 4 E4B via Ollama vision API.
Writes `ocr.md`, `chart.png`, and `metadata.json` to `output/{stem}/`.

In [3]:
stem_dir = run_step1(PDF_PATH, OUTPUT_DIR, ollama_url=OLLAMA_URL, force=FORCE_RUN)
print(f"\nStep 1 complete → {stem_dir}")

# Preview OCR markdown
ocr_md = (stem_dir / "ocr.md").read_text(encoding="utf-8")
print(f"\n--- ocr.md preview (first 500 chars) ---")
print(ocr_md[:500])

# Pretty-print metadata
import json
metadata = json.loads((stem_dir / "metadata.json").read_text(encoding="utf-8"))
print(f"\n--- metadata.json ---")
print(json.dumps(metadata, indent=2, ensure_ascii=False)[:1000])

[run_step1] PAGASA_25-TC22_Verbena_TCB#24: wrote ocr.md (7680 chars)
[run_step1] PAGASA_25-TC22_Verbena_TCB#24: wrote forecast_table.md (1391 chars)
[run_step1] PAGASA_25-TC22_Verbena_TCB#24: saved chart.png (page 0)
[run_step1] PAGASA_25-TC22_Verbena_TCB#24: wrote metadata.json

Step 1 complete → 10-etl-e2e/output/PAGASA_25-TC22_Verbena_TCB#24

--- ocr.md preview (first 500 chars) ---
<!-- Page 1 -->

TROPICAL CYCLONE BULLETIN NO. 24
VERBENA

"VERBENA" WEAKENS WHILE MOVING WEST SOUTHWESTWARD SLOWLY

**General Remarks**
The center of Typhoon VERBENA was estimated based on all available data at 270 km Northwest of Pag-asa Island, Kalayaan, Palawan (OUTSIDE PAR).

**Forecast Track**

| Date and Time | Lat. ($\text{N}^{\circ}$) | Lon. ($\text{E}^{\circ}$) |
| :--- | :---: | :---: |
| 12-Hour Forecast 21 Nov 2025 | 12.9 | 112.6 |
| 24-Hour Forecast 22 Nov 2025 | 13.0 | 112.1 |
| 3

--- metadata.json ---
{
  "bulletin_type": "TCA",
  "storm": {
    "name": "VERBENA",
    "category": "Typhoo

## Step 2: Radio Scripts + TTS Text

Generates a spoken weather announcement and TTS-optimised plain text for each language.
Writes `radio_{lang}.md` and `tts_{lang}.txt` to `output/{stem}/`.

In [4]:
for lang in LANGUAGES:
    radio_path = run_step2(STEM, lang, OUTPUT_DIR, ollama_url=OLLAMA_URL, force=FORCE_RUN)
    print(f"\n{'='*60}")
    print(f"[{lang.upper()}] {radio_path.name}")
    print('='*60)
    print(radio_path.read_text(encoding="utf-8")[:400])
print("\nStep 2 complete.")

[run_step2] PAGASA_25-TC22_Verbena_TCB#24/ceb: using hybrid input (metadata + OCR)
[run_step2] PAGASA_25-TC22_Verbena_TCB#24/ceb: wrote radio + tts files

[CEB] radio_ceb.md
Ati, pag-amping mo kaayo. Si Bagyong VERBENA ang atong kasaligan. Kanunay kining hinay nga naglihok paingon sa amihanang-kasadlanan paingon sa kasadlanan. Wala pa kini'y signal sa hangin. Pero, dako kaayo ang posibleng pag-ulan.

Magbantay mo sa mga lugar nga dunay pasidaan sa pag-ulan, ilabi na sa Central Luzon ug CALABARZON. Tungko pa gihapon ang pag-andam sa pag-alis sa baybayon.

Kay bisan uns
[run_step2] PAGASA_25-TC22_Verbena_TCB#24/tl: using hybrid input (metadata + OCR)
[run_step2] PAGASA_25-TC22_Verbena_TCB#24/tl: wrote radio + tts files

[TL] radio_tl.md
Typhoon VERBENA, humihina habang pa-west-southwest itong gumagalaw nang mabagal. Kasalukuyan itong nasa lugar na matatagpuan sa Palawan. Patuloy itong liliko pa-west-southwest. May babala para sa malalakas na ulan sa mga lugar sa Central Luzon at CALABARZ

## Step 3: TTS Synthesis → MP3

Synthesizes MP3 audio for each language using:
- English: Coqui XTTS v2 (speaker: Damien Black)
- Tagalog / Cebuano: Facebook MMS VITS

Writes `audio_{lang}.mp3` to `output/{stem}/`.

In [5]:
for lang in LANGUAGES:
    mp3_path = run_step3(STEM, lang, OUTPUT_DIR, TTS_MODELS_DIR, force=FORCE_RUN)
    size_kb = mp3_path.stat().st_size // 1024
    print(f"[{lang.upper()}] {mp3_path.name}  ({size_kb} KB)")
print("\nStep 3 complete.")

/Users/josereyes/Dev/gemma4-hackathon/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[MMSSynthesizer] loading facebook/mms-tts-ceb on cpu
[run_step3] PAGASA_25-TC22_Verbena_TCB#24/ceb: wrote 10-etl-e2e/output/PAGASA_25-TC22_Verbena_TCB#24/audio_ceb.mp3
[CEB] audio_ceb.mp3  (935 KB)
[MMSSynthesizer] loading facebook/mms-tts-tgl on cpu
[run_step3] PAGASA_25-TC22_Verbena_TCB#24/tl: wrote 10-etl-e2e/output/PAGASA_25-TC22_Verbena_TCB#24/audio_tl.mp3
[TL] audio_tl.mp3  (59 KB)
[CoquiXTTSSynthesizer] loading on cpu
[run_step3] PAGASA_25-TC22_Verbena_TCB#24/en: wrote 10-etl-e2e/output/PAGASA_25-TC22_Verbena_TCB#24/audio_en.mp3
[EN] audio_en.mp3  (702 KB)

Step 3 complete.


In [6]:
from IPython.display import Audio, display, Markdown

for lang in LANGUAGES:
    mp3_path = OUTPUT_DIR / STEM / f"audio_{lang}.mp3"
    if mp3_path.exists():
        display(Markdown(f"**{lang.upper()}**"))
        display(Audio(str(mp3_path)))

**CEB**

**TL**

**EN**

## Output Summary

In [7]:
stem_dir = OUTPUT_DIR / STEM
print(f"Artefacts in {stem_dir.resolve()}:\n")
for f in sorted(stem_dir.iterdir()):
    size = f.stat().st_size
    unit = 'KB' if size >= 1024 else 'B'
    val = size // 1024 if size >= 1024 else size
    print(f"  {f.name:<35}  {val:>6} {unit}")

Artefacts in /Users/josereyes/Dev/gemma4-hackathon/notebooks/10-etl-e2e/output/PAGASA_25-TC22_Verbena_TCB#24:

  audio_ceb.mp3                           935 KB
  audio_en.mp3                            702 KB
  audio_tl.mp3                             59 KB
  chart.png                               687 KB
  forecast_table.md                         1 KB
  metadata.json                             1 KB
  ocr.md                                    7 KB
  radio_ceb.md                            756 B
  radio_en.md                             638 B
  radio_tl.md                             581 B
  tts_ceb.txt                             766 B
  tts_en.txt                              571 B
  tts_tl.txt                               52 B
